# Matrix Multiplication FPGA Test

Load `matmul_basic_overlay.bit`, run `(batch x 64) @ (64 x 64)` on the FPGA, and compare each result with NumPy.

In [ ]:
import time

import numpy as np
from pynq import Overlay, allocate


BATCH_SIZES = (1, 32, 512, 2048)
INPUT_SIZE = 64
OUTPUT_SIZE = 64
DTYPE = np.int32

In [ ]:
overlay = Overlay("matmul_basic_overlay.bit")
ip = overlay.matmul_0
overlay.ip_dict

In [ ]:
def write_address(ip, offset, address):
    ip.write(offset, int(address) & 0xFFFFFFFF)
    ip.write(offset + 4, (int(address) >> 32) & 0xFFFFFFFF)


def run_fpga_matmul(ip, activations_np, weights_np):
    batch_size = activations_np.shape[0]
    activations = allocate(shape=activations_np.shape, dtype=DTYPE)
    weights = allocate(shape=weights_np.shape, dtype=DTYPE)
    output = allocate(shape=(batch_size, OUTPUT_SIZE), dtype=DTYPE)

    np.copyto(activations, activations_np)
    np.copyto(weights, weights_np)
    output[:] = 0

    activations.sync_to_device()
    weights.sync_to_device()
    output.sync_to_device()

    write_address(ip, 0x10, activations.physical_address)
    write_address(ip, 0x1C, weights.physical_address)
    write_address(ip, 0x28, output.physical_address)
    ip.write(0x34, batch_size)

    start = time.perf_counter()
    ip.write(0x00, 1)
    while (ip.read(0x00) & 0x2) == 0:
        pass
    elapsed = time.perf_counter() - start

    output.sync_from_device()
    result = np.array(output)

    activations.freebuffer()
    weights.freebuffer()
    output.freebuffer()

    return result, elapsed

In [ ]:
rng = np.random.default_rng(42)
results = []

for batch_size in BATCH_SIZES:
    activations = rng.integers(-8, 8, size=(batch_size, INPUT_SIZE), dtype=DTYPE)
    weights = rng.integers(-8, 8, size=(INPUT_SIZE, OUTPUT_SIZE), dtype=DTYPE)

    cpu_start = time.perf_counter()
    expected = activations @ weights
    cpu_elapsed = time.perf_counter() - cpu_start

    fpga_result, fpga_elapsed = run_fpga_matmul(ip, activations, weights)
    max_error = int(np.max(np.abs(fpga_result - expected)))

    results.append(
        {
            "batch": batch_size,
            "cpu_time": cpu_elapsed,
            "fpga_time": fpga_elapsed,
            "max_error": max_error,
            "checksum": int(fpga_result.sum()),
        }
    )

results